# SOTA Recommender Comparison

Compare your two-tower model against state-of-the-art recommendation approaches:

1. **Classic Methods**: ALS, BPR, Item-KNN
2. **Hybrid Methods**: LightFM 
3. **Graph Methods**: LightGCN (via RecBole)
4. **External APIs**: Recombee (free trial)

## Installation
```bash
pip install implicit lightfm recbole torch faiss-cpu recombee-api-client
```

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from collections import defaultdict
import time
import warnings
warnings.filterwarnings('ignore')

from src.data.processor import DataProcessor
from src.utils.config import load_config
from src.evaluation.metrics import (
    precision_at_k, recall_at_k, ndcg_at_k, hit_rate_at_k, mrr_at_k
)

print("Setup complete!")

## Load and Prepare Data

In [ ]:
# Load data
config = load_config()
processor = DataProcessor(config)

items_df, events_df = processor.load_data()
interactions_df = processor.build_interactions()
items_df, interactions_df = processor.encode_features()

# Split data
train_df, val_df, test_df = processor.prepare_datasets(temporal_split=True)

# Prepare ground truth
test_ground_truth = {}
for user_idx, group in test_df.groupby('user_idx'):
    test_ground_truth[user_idx] = set(group['item_idx'].values)

train_user_items = {}
for user_idx, group in train_df.groupby('user_idx'):
    train_user_items[user_idx] = set(group['item_idx'].values)

num_users = processor.vocab_sizes['user']
num_items = processor.vocab_sizes['item']

print(f"Users: {num_users}, Items: {num_items}")
print(f"Train: {len(train_df)}, Test users: {len(test_ground_truth)}")

In [ ]:
def evaluate_model(recommendations, ground_truth, train_items, name, k_values=[5, 10, 20, 50]):
    """Comprehensive evaluation of a recommendation model."""
    results = {'model': name}
    
    for k in k_values:
        hits, precisions, recalls, ndcgs, mrrs = [], [], [], [], []
        
        for user_idx, recs in recommendations.items():
            if user_idx not in ground_truth or len(ground_truth[user_idx]) == 0:
                continue
            
            relevant = ground_truth[user_idx]
            hits.append(hit_rate_at_k(recs, relevant, k))
            precisions.append(precision_at_k(recs, relevant, k))
            recalls.append(recall_at_k(recs, relevant, k))
            ndcgs.append(ndcg_at_k(recs, relevant, k))
            mrrs.append(mrr_at_k(recs, relevant, k))
        
        results[f'HR@{k}'] = np.mean(hits) if hits else 0
        results[f'P@{k}'] = np.mean(precisions) if precisions else 0
        results[f'R@{k}'] = np.mean(recalls) if recalls else 0
        results[f'NDCG@{k}'] = np.mean(ndcgs) if ndcgs else 0
        results[f'MRR@{k}'] = np.mean(mrrs) if mrrs else 0
    
    results['n_users'] = len([u for u in recommendations if u in ground_truth])
    return results

all_results = []

## 1. Baseline: Most Popular

In [ ]:
print("=" * 60)
print("1. POPULARITY BASELINE")
print("=" * 60)

item_popularity = train_df.groupby('item_idx').size().sort_values(ascending=False)
popular_items = item_popularity.index.tolist()

pop_recs = {}
for user_idx in test_ground_truth:
    exclude = train_user_items.get(user_idx, set())
    pop_recs[user_idx] = [i for i in popular_items if i not in exclude][:50]

pop_results = evaluate_model(pop_recs, test_ground_truth, train_user_items, 'Popularity')
all_results.append(pop_results)
print(f"HR@10: {pop_results['HR@10']:.4f}, NDCG@10: {pop_results['NDCG@10']:.4f}")

## 2. ALS (Alternating Least Squares)

In [ ]:
print("\n" + "=" * 60)
print("2. ALS (Alternating Least Squares)")
print("=" * 60)

try:
    import implicit
    from scipy.sparse import csr_matrix
    
    # Build sparse matrix
    rows = train_df['user_idx'].values
    cols = train_df['item_idx'].values
    data = train_df['interaction_strength'].values
    user_item = csr_matrix((data, (rows, cols)), shape=(num_users, num_items))
    
    # Try different configurations
    best_als_hr = 0
    best_als_config = None
    
    for factors in [32, 64, 128]:
        for reg in [0.01, 0.1, 0.5]:
            als_model = implicit.als.AlternatingLeastSquares(
                factors=factors,
                regularization=reg,
                iterations=20,
                random_state=42,
                use_gpu=False
            )
            als_model.fit(user_item, show_progress=False)
            
            # Quick eval on subset
            sample_users = list(test_ground_truth.keys())[:200]
            als_recs_sample = {}
            for user_idx in sample_users:
                if user_idx < user_item.shape[0]:
                    ids, _ = als_model.recommend(
                        user_idx, user_item[user_idx], N=50,
                        filter_already_liked_items=True
                    )
                    als_recs_sample[user_idx] = ids.tolist()
            
            temp_results = evaluate_model(als_recs_sample, test_ground_truth, train_user_items, 'temp')
            if temp_results['HR@10'] > best_als_hr:
                best_als_hr = temp_results['HR@10']
                best_als_config = (factors, reg)
    
    print(f"Best config: factors={best_als_config[0]}, reg={best_als_config[1]}")
    
    # Train final model with best config
    als_model = implicit.als.AlternatingLeastSquares(
        factors=best_als_config[0],
        regularization=best_als_config[1],
        iterations=30,
        random_state=42
    )
    als_model.fit(user_item, show_progress=False)
    
    als_recs = {}
    for user_idx in test_ground_truth:
        if user_idx < user_item.shape[0]:
            ids, _ = als_model.recommend(
                user_idx, user_item[user_idx], N=50,
                filter_already_liked_items=True
            )
            als_recs[user_idx] = ids.tolist()
    
    als_results = evaluate_model(als_recs, test_ground_truth, train_user_items, 'ALS')
    all_results.append(als_results)
    print(f"HR@10: {als_results['HR@10']:.4f}, NDCG@10: {als_results['NDCG@10']:.4f}")
    
except ImportError:
    print("Install: pip install implicit")

## 3. BPR (Bayesian Personalized Ranking)

In [ ]:
print("\n" + "=" * 60)
print("3. BPR (Bayesian Personalized Ranking)")
print("=" * 60)

try:
    bpr_model = implicit.bpr.BayesianPersonalizedRanking(
        factors=64,
        learning_rate=0.1,
        regularization=0.01,
        iterations=100,
        random_state=42
    )
    bpr_model.fit(user_item, show_progress=False)
    
    bpr_recs = {}
    for user_idx in test_ground_truth:
        if user_idx < user_item.shape[0]:
            ids, _ = bpr_model.recommend(
                user_idx, user_item[user_idx], N=50,
                filter_already_liked_items=True
            )
            bpr_recs[user_idx] = ids.tolist()
    
    bpr_results = evaluate_model(bpr_recs, test_ground_truth, train_user_items, 'BPR')
    all_results.append(bpr_results)
    print(f"HR@10: {bpr_results['HR@10']:.4f}, NDCG@10: {bpr_results['NDCG@10']:.4f}")
    
except Exception as e:
    print(f"BPR failed: {e}")

## 4. LightFM (Hybrid Model)

In [ ]:
print("\n" + "=" * 60)
print("4. LightFM (Hybrid Collaborative + Content)")
print("=" * 60)

try:
    from lightfm import LightFM
    from lightfm.data import Dataset as LFMDataset
    from scipy.sparse import coo_matrix
    
    # Build item features
    item_features_list = []
    for _, row in items_df.iterrows():
        features = [
            f"cat:{row['category_idx']}",
            f"brand:{row['brand_idx']}",
            f"cond:{row['condition_idx']}"
        ]
        item_features_list.append((row['item_idx'], features))
    
    # Create dataset
    lfm_dataset = LFMDataset()
    lfm_dataset.fit(
        users=range(num_users),
        items=range(num_items),
        item_features=[f"cat:{i}" for i in range(200)] + 
                      [f"brand:{i}" for i in range(200)] +
                      [f"cond:{i}" for i in range(20)]
    )
    
    interactions, _ = lfm_dataset.build_interactions(
        [(r['user_idx'], r['item_idx'], r['interaction_strength']) 
         for _, r in train_df.iterrows()]
    )
    
    item_features = lfm_dataset.build_item_features(item_features_list)
    
    # Test different losses
    for loss in ['warp', 'bpr']:
        lfm_model = LightFM(
            loss=loss,
            no_components=64,
            learning_rate=0.05,
            item_alpha=1e-6,
            user_alpha=1e-6,
            random_state=42
        )
        lfm_model.fit(
            interactions,
            item_features=item_features,
            epochs=30,
            num_threads=4,
            verbose=False
        )
        
        n_users_lfm, n_items_lfm = interactions.shape
        lfm_recs = {}
        
        for user_idx in test_ground_truth:
            if user_idx < n_users_lfm:
                scores = lfm_model.predict(
                    user_idx, 
                    np.arange(n_items_lfm),
                    item_features=item_features
                )
                exclude = train_user_items.get(user_idx, set())
                scores[list(exclude)] = -np.inf
                top_items = np.argsort(-scores)[:50]
                lfm_recs[user_idx] = top_items.tolist()
        
        lfm_results = evaluate_model(lfm_recs, test_ground_truth, train_user_items, f'LightFM-{loss}')
        all_results.append(lfm_results)
        print(f"LightFM ({loss}): HR@10: {lfm_results['HR@10']:.4f}, NDCG@10: {lfm_results['NDCG@10']:.4f}")

except ImportError:
    print("Install: pip install lightfm")

## 5. Item-KNN (Nearest Neighbor)

In [ ]:
print("\n" + "=" * 60)
print("5. Item-KNN (Cosine Similarity)")
print("=" * 60)

try:
    # Item-based KNN using implicit
    knn_model = implicit.nearest_neighbours.CosineRecommender(K=50)
    knn_model.fit(user_item.T)  # Item-item similarity
    
    knn_recs = {}
    for user_idx in test_ground_truth:
        if user_idx < user_item.shape[0]:
            ids, _ = knn_model.recommend(
                user_idx, user_item[user_idx], N=50,
                filter_already_liked_items=True
            )
            knn_recs[user_idx] = ids.tolist()
    
    knn_results = evaluate_model(knn_recs, test_ground_truth, train_user_items, 'ItemKNN')
    all_results.append(knn_results)
    print(f"HR@10: {knn_results['HR@10']:.4f}, NDCG@10: {knn_results['NDCG@10']:.4f}")
    
except Exception as e:
    print(f"Item-KNN failed: {e}")

## 6. RecBole (Deep Learning Models)

In [ ]:
print("\n" + "=" * 60)
print("6. RecBole Deep Learning Models")
print("=" * 60)

try:
    from recbole.quick_start import run_recbole
    from recbole.config import Config as RBConfig
    from recbole.data import create_dataset, data_preparation
    from recbole.utils import init_seed, init_logger
    import tempfile
    import os
    
    # Prepare data in RecBole format
    with tempfile.TemporaryDirectory() as tmpdir:
        dataset_name = 'marketplace'
        os.makedirs(f'{tmpdir}/{dataset_name}', exist_ok=True)
        
        # Create .inter file
        inter_data = train_df[['user_idx', 'item_idx', 'interaction_strength']].copy()
        inter_data.columns = ['user_id:token', 'item_id:token', 'rating:float']
        inter_data.to_csv(f'{tmpdir}/{dataset_name}/{dataset_name}.inter', sep='\t', index=False)
        
        # Run LightGCN
        config_dict = {
            'model': 'LightGCN',
            'data_path': tmpdir,
            'dataset': dataset_name,
            'embedding_size': 64,
            'n_layers': 3,
            'epochs': 20,
            'train_batch_size': 2048,
            'eval_batch_size': 4096,
            'learning_rate': 0.001,
            'topk': [5, 10, 20],
            'metrics': ['Hit', 'NDCG', 'Precision', 'Recall'],
            'valid_metric': 'Hit@10',
            'show_progress': False
        }
        
        result = run_recbole(config_dict=config_dict)
        print(f"\nLightGCN Results:")
        for metric, value in result['test_result'].items():
            print(f"  {metric}: {value:.4f}")
            
except ImportError:
    print("RecBole not installed. Install with: pip install recbole")
except Exception as e:
    print(f"RecBole failed: {e}")
    print("This is expected if data format doesn't match. Skipping...")

## 7. External API: Recombee (Optional)

Recombee offers a free 30-day trial. Sign up at https://www.recombee.com/

In [ ]:
print("\n" + "=" * 60)
print("7. Recombee API (External SOTA)")
print("=" * 60)

# Set your Recombee credentials here
RECOMBEE_DB = None  # 'your-database-name'
RECOMBEE_TOKEN = None  # 'your-secret-token'

if RECOMBEE_DB and RECOMBEE_TOKEN:
    try:
        from recombee_api_client.api_client import RecombeeClient
        from recombee_api_client import api_requests
        
        client = RecombeeClient(RECOMBEE_DB, RECOMBEE_TOKEN)
        
        # Add items (do once)
        print("Adding items to Recombee...")
        item_values = []
        for _, row in items_df.head(1000).iterrows():  # Limit for demo
            item_values.append(
                api_requests.SetItemValues(
                    str(row['item_idx']),
                    {
                        'category': str(row['category_idx']),
                        'brand': str(row['brand_idx']),
                        'price': float(row['price_normalized'])
                    },
                    cascade_create=True
                )
            )
        client.send(api_requests.Batch(item_values))
        
        # Add interactions
        print("Adding interactions...")
        interactions_batch = []
        for _, row in train_df.head(10000).iterrows():
            interactions_batch.append(
                api_requests.AddPurchase(
                    str(row['user_idx']),
                    str(row['item_idx']),
                    cascade_create=True
                )
            )
        client.send(api_requests.Batch(interactions_batch))
        
        # Get recommendations
        print("Getting recommendations...")
        recombee_recs = {}
        for user_idx in list(test_ground_truth.keys())[:100]:
            try:
                response = client.send(
                    api_requests.RecommendItemsToUser(
                        str(user_idx),
                        50,
                        return_properties=False
                    )
                )
                recombee_recs[user_idx] = [int(r['id']) for r in response['recomms']]
            except:
                pass
        
        if recombee_recs:
            recombee_results = evaluate_model(recombee_recs, test_ground_truth, train_user_items, 'Recombee')
            all_results.append(recombee_results)
            print(f"HR@10: {recombee_results['HR@10']:.4f}, NDCG@10: {recombee_results['NDCG@10']:.4f}")
        
    except ImportError:
        print("Install: pip install recombee-api-client")
    except Exception as e:
        print(f"Recombee failed: {e}")
else:
    print("Recombee credentials not set. Sign up at https://www.recombee.com/")
    print("Then set RECOMBEE_DB and RECOMBEE_TOKEN above.")

## 8. Your Current Two-Tower Model

In [ ]:
print("\n" + "=" * 60)
print("8. Your Current Two-Tower Model")
print("=" * 60)

# Add your model results from training
two_tower_results = {
    'model': 'TwoTower (Current)',
    'HR@5': 0.0081,
    'HR@10': 0.0146,
    'HR@20': 0.0227,
    'HR@50': 0.0474,
    'P@5': 0.0016,
    'P@10': 0.0015,
    'P@20': 0.0013,
    'P@50': 0.0012,
    'R@5': 0.0014,
    'R@10': 0.0026,
    'R@20': 0.0042,
    'R@50': 0.0096,
    'NDCG@5': 0.0016,
    'NDCG@10': 0.0021,
    'NDCG@20': 0.0025,
    'NDCG@50': 0.0041,
    'MRR@5': 0.0027,
    'MRR@10': 0.0036,
    'MRR@20': 0.0042,
    'MRR@50': 0.0049,
    'n_users': 1983
}
all_results.append(two_tower_results)
print(f"HR@10: {two_tower_results['HR@10']:.4f}, NDCG@10: {two_tower_results['NDCG@10']:.4f}")

## Final Comparison

In [ ]:
print("\n" + "=" * 70)
print("FINAL COMPARISON - ALL MODELS")
print("=" * 70)

results_df = pd.DataFrame(all_results)
results_df = results_df.set_index('model')

# Select key metrics
key_metrics = ['HR@5', 'HR@10', 'HR@20', 'NDCG@10', 'MRR@10', 'n_users']
display_df = results_df[[c for c in key_metrics if c in results_df.columns]].copy()

# Sort by HR@10
display_df = display_df.sort_values('HR@10', ascending=False)

print(display_df.round(4).to_string())

# Highlight best model
print("\n" + "=" * 70)
best_model = display_df['HR@10'].idxmax()
best_score = display_df['HR@10'].max()
your_score = two_tower_results['HR@10']

print(f"BEST MODEL: {best_model} (HR@10 = {best_score:.4f})")
print(f"Your TwoTower: HR@10 = {your_score:.4f}")

if your_score < best_score:
    improvement = (best_score - your_score) / your_score * 100
    print(f"\nPOTENTIAL IMPROVEMENT: {improvement:.1f}% by switching to {best_model}")

In [ ]:
# Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# HR@K comparison
hr_cols = [c for c in results_df.columns if c.startswith('HR@')]
hr_df = results_df[hr_cols].T
hr_df.index = [int(c.split('@')[1]) for c in hr_df.index]
hr_df = hr_df.sort_index()

for model in hr_df.columns:
    axes[0].plot(hr_df.index, hr_df[model], marker='o', label=model)

axes[0].set_xlabel('K')
axes[0].set_ylabel('Hit Rate')
axes[0].set_title('Hit Rate @ K Comparison')
axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[0].grid(True, alpha=0.3)

# Bar chart for HR@10
hr10 = results_df['HR@10'].sort_values(ascending=True)
colors = ['red' if m == 'TwoTower (Current)' else 'steelblue' for m in hr10.index]
axes[1].barh(range(len(hr10)), hr10.values, color=colors)
axes[1].set_yticks(range(len(hr10)))
axes[1].set_yticklabels(hr10.index)
axes[1].set_xlabel('Hit Rate @ 10')
axes[1].set_title('Model Comparison (HR@10)')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Recommendations

In [ ]:
print("\n" + "=" * 70)
print("RECOMMENDATIONS")
print("=" * 70)

recs = """
Based on the comparison:

1. IMMEDIATE WINS:
   - If ALS/BPR beats your model -> Use those as they're faster and simpler
   - If LightFM beats your model -> Hybrid approach works better for your data

2. FOR PRODUCTION:
   - Consider Recombee or similar APIs if budget allows
   - They handle cold-start, scaling, and tuning automatically

3. TO FIX YOUR TWO-TOWER MODEL:
   a) Data Issues:
      - Use leave-one-out evaluation instead of temporal split
      - Ensure test users have training history
   
   b) Model Issues:
      - Increase dropout (0.4-0.5)
      - Add L2 regularization (1e-4)
      - Increase embedding dim (64-128)
      - More negative samples (10-20)
   
   c) Training Issues:
      - Use cosine annealing learning rate
      - Add batch normalization
      - Use sampled softmax for large item sets

4. ADVANCED OPTIONS:
   - LightGCN (graph neural network)
   - SASRec (sequential model)
   - BERT4Rec (transformer-based)
"""
print(recs)

## Save Results

In [ ]:
# Save comparison results
results_df.to_csv('sota_comparison_results.csv')
print("Results saved to sota_comparison_results.csv")